In [1]:
import yfinance as yf
import pandas as pd
import lightgbm as lgb
import shap
import hmmlearn

print("all good")

# quick data pull test
spy = yf.download("SPY", start="2020-01-01", end="2020-12-31")
print(spy.tail(3))

all good


[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open    Volume
Ticker             SPY         SPY         SPY         SPY       SPY
Date                                                                
2020-12-28  346.465637  346.856613  345.441604  346.065314  39000400
2020-12-29  345.804749  348.169329  345.218256  347.992449  53680500
2020-12-30  346.298096  347.331447  345.907119  346.623928  49455300


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

# create data folder if it doesn't exist
os.makedirs("../data", exist_ok=True)
print("imports done")

imports done


In [3]:
# Wikipedia blocks plain requests - use requests library with a browser header instead
import requests

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

response = requests.get(url, headers=headers)
sp500_table = pd.read_html(response.text)[0]
tickers = sp500_table["Symbol"].tolist()
tickers = [t.replace(".", "-") for t in tickers]

print(f"got {len(tickers)} tickers")
print(tickers[:5])

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17500\3972865825.py:8: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500_table = pd.read_html(response.text)[0]


got 503 tickers
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN']


In [4]:
raw = yf.download(
    tickers,
    start="2019-01-01",
    end="2024-12-31",
    auto_adjust=True
)

print(raw.shape)
print(raw.head(3))

[**************        30%                       ]  153 of 503 completed$Q: possibly delisted; no price data found  (1d 2019-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1546318800, endDate = 1735621200")
[**********************51%                       ]  259 of 503 completed$SNDK: possibly delisted; no price data found  (1d 2019-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1546318800, endDate = 1735621200")
[**********************96%*********************  ]  483 of 503 completed$FDXF: possibly delisted; no price data found  (1d 2019-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1546318800, endDate = 1735621200")
[*********************100%***********************]  503 of 503 completed

3 Failed downloads:
['Q', 'SNDK', 'FDXF']: possibly delisted; no price data found  (1d 2019-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1546318800, endDate = 1735621200")


(1509, 2518)
Price      Adj Close               Close                             \
Ticker          FDXF   Q SNDK          A       AAPL       ABBV ABNB   
Date                                                                  
2019-01-02       NaN NaN  NaN  62.300140  37.469208  64.789444  NaN   
2019-01-03       NaN NaN  NaN  60.005009  33.737000  62.654713  NaN   
2019-01-04       NaN NaN  NaN  62.082001  35.177200  64.673256  NaN   

Price                                         ...   Volume                    \
Ticker            ABT       ACGL         ACN  ...       WY     WYNN      XEL   
Date                                          ...                              
2019-01-02  60.801231  24.904034  126.118225  ...  7442000  4174400  4476100   
2019-01-03  57.931744  24.514164  121.812340  ...  9788300  2885100  5287600   
2019-01-04  59.585194  25.094212  126.548836  ...  5843900  3007200  5535600   

Price                                                                       
Ti

In [5]:
raw.to_parquet("../data/sp500_raw.parquet")
print("saved to data/sp500_raw.parquet")

saved to data/sp500_raw.parquet


In [6]:
import os
size = os.path.getsize("../data/sp500_raw.parquet")
print(f"file size: {size / 1024 / 1024:.1f} MB")

file size: 29.8 MB
